# Cost function

_Machine Learning_

**Add up the loss over every example. That total is what we shrink.**

Loss scores one example. Cost scores the whole dataset.

     Just add the loss of every example together.

---

This notebook builds the **cost function** one piece at a time — first the loss of a single example, then the total over a whole dataset, then the bowl-shaped curve we slide down. Run each cell, read the note above it, and watch where the cost bottoms out. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — NumPy

### Step 1 — Make a tiny dataset and define the loss

We generate 20 points that lie roughly on the line `y = 2x + 1` (with a little noise). The **cost** `J` adds up the squared loss of every example, then averages it: for each point we take the prediction error, square it, halve it, and take the mean. The halving is a convention that makes the derivative clean later.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = rng.uniform(0, 10, size=20)
y = 2.0 * X + 1.0 + rng.normal(0, 1, size=20)   # true line y = 2x + 1

def cost(theta, b):
    pred = theta * X + b               # model's prediction for every point
    errors = pred - y                  # how far off each prediction is
    return np.mean(0.5 * errors ** 2)  # mean squared loss = J

### Step 2 — Compare a bad guess to the true line

The cost is just a number that scores how well a `(theta, b)` pair fits the data. A bad guess (`theta=0, b=0`, a flat line through the origin) should score a large cost; the true parameters (`theta=2, b=1`) should score a small one. Lower cost means a better fit.

In [ ]:
bad_cost = cost(0.0, 0.0)    # flat line through the origin — a poor fit
good_cost = cost(2.0, 1.0)   # the true parameters — a good fit

print("cost at bad   theta=0, b=0 :", round(bad_cost, 3))
print("cost at good  theta=2, b=1 :", round(good_cost, 3))

### Step 3 — Scan the slope to reveal the bowl

Hold the intercept fixed at `b=1` and sweep the slope `theta` across a few values. As `theta` moves toward the true value of `2`, the cost falls and then rises again — the cost function is a **bowl with a single bottom**, and that bottom is the fit we want.

In [ ]:
# scan theta -> J(theta) is a bowl with one bottom
for t in [0.0, 1.0, 2.0, 3.0]:
    J = cost(t, 1.0)
    print("theta=%.1f  J=%.3f" % (t, J))

## Visualize the data & results

_On real diabetes data, as we vary the BMI slope theta, where does total squared cost bottom out?_

### Step 1 — Build the cost over real diabetes data

Now use a real dataset: predict disease progression from the **BMI** feature alone. We center the BMI around its mean and anchor predictions at the mean target, so a slope of `theta` swings the line up and down through the cloud. The same averaged squared-loss `cost(theta)` scores each slope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# real diabetes data: predict progression from BMI alone
db = load_diabetes()
x = db.data[:, 2]              # BMI feature (already centered/scaled)
y = db.target
xbar = x.mean()
ybar = y.mean()

def cost(theta):
    pred = theta * (x - xbar) + ybar   # line pivoting through the data's center
    errors = pred - y
    return np.mean(0.5 * errors ** 2)  # same mean squared loss J

### Step 2 — Evaluate the cost across many slopes

We sweep `theta` over a wide range and record `J(theta)` at each value. Collecting these points traces out the cost curve so we can see exactly where it bottoms out.

In [ ]:
thetas = np.linspace(0, 2000, 25)
J = [cost(t) for t in thetas]

### Step 3 — Plot the cost curve

Plotting `J` against `theta` shows the bowl on real data. The lowest point of the curve is the BMI slope that minimizes total squared error — the best-fitting line for this single feature.

In [ ]:
plt.plot(thetas, J, color="#4ea1ff", label="J(theta)")
plt.xlabel("theta (slope on BMI)")
plt.ylabel("mean squared loss J")
plt.title("Cost J(theta) over the diabetes dataset")
plt.legend()
plt.show()